# Duplex KV Cache Walkthrough

Run this notebook cell by cell in Google Colab to understand the system prompt update cache behavior without loading the full MiniCPM-o model first.

The first half uses tiny tensors so the cache is visible. The second half imports the repo's real `StreamDecoder.update_system_prompt()` method and runs it with a fake LLM, matching the implementation path used by the tests.

Mental model for the duplex cache layout:

```text
[system prefix + optional ref audio] [previous summary] [system suffix] [live conversation units]
```

When the system prompt changes, the implementation rebuilds the protected system span and keeps later unit cache entries. If the new system span has a different length, preserved unit keys are RoPE-reindexed so their absolute positions move to the new location.

## 1. Colab Setup

If you cloned the repo into `/content/MiniCPM-o-Demo`, this cell should work as-is. If your checkout lives elsewhere, change `REPO_PATH`.

In [ ]:
%pip -q install transformers pytest

In [ ]:
from pathlib import Path
import sys

REPO_PATH = Path('/content/MiniCPM-o-Demo')

if REPO_PATH.exists():
    sys.path.insert(0, str(REPO_PATH / 'src'))
    print('Repo found:', REPO_PATH)
else:
    print('Repo not found yet. Clone/upload it, or update REPO_PATH.')
    print('Example: !git clone --branch YOUR_BRANCH https://github.com/YOUR_ORG/YOUR_REPO.git /content/MiniCPM-o-Demo')

In [ ]:
import torch
from types import SimpleNamespace

torch.set_printoptions(precision=2, sci_mode=False)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

## 2. What A KV Cache Stores

A transformer layer stores keys and values for tokens it already processed. A common shape is:

```text
key/value: [batch, heads, sequence_length, head_dim]
```

This toy cache uses readable numbers instead of real attention vectors.

In [ ]:
def make_visible_cache(token_ids, start_position=0, heads=1, head_dim=4):
    """Create a readable fake KV cache for one layer.

    Keys encode token_id + absolute position.
    Values encode token_id - absolute position.
    Real models use learned projections, but the cache shape and indexing idea is the same.
    """
    token_ids = torch.tensor(token_ids, dtype=torch.float32)
    positions = torch.arange(start_position, start_position + len(token_ids), dtype=torch.float32)
    base = token_ids[:, None].repeat(1, head_dim)
    pos = positions[:, None].repeat(1, head_dim)
    keys = (base + pos / 100).reshape(1, heads, len(token_ids), head_dim)
    values = (base - pos / 100).reshape(1, heads, len(token_ids), head_dim)
    return ((keys, values),)

def cache_len(cache):
    if cache is None:
        return 0
    return cache[0][0].shape[2]

def slice_cache(cache, start, end=None):
    return tuple((k[:, :, start:end, :].clone(), v[:, :, start:end, :].clone()) for k, v in cache)

def concat_caches(*parts):
    parts = [p for p in parts if p is not None and cache_len(p) > 0]
    if not parts:
        return None
    layers = []
    for layer_index in range(len(parts[0])):
        keys = torch.cat([p[layer_index][0] for p in parts], dim=2)
        values = torch.cat([p[layer_index][1] for p in parts], dim=2)
        layers.append((keys, values))
    return tuple(layers)

def show_cache(cache, labels=None, title='cache'):
    keys, values = cache[0]
    print(f'\n{title}: length={cache_len(cache)}, shape={tuple(keys.shape)}')
    for i in range(cache_len(cache)):
        label = labels[i] if labels and i < len(labels) else str(i)
        k = keys[0, 0, i, :].tolist()
        v = values[0, 0, i, :].tolist()
        print(f'{i:02d} {label:<12} key={k} value={v}')

system_tokens = [10, 11, 12]
unit_tokens = [70, 71]
cache = concat_caches(
    make_visible_cache(system_tokens, start_position=0),
    make_visible_cache(unit_tokens, start_position=len(system_tokens)),
)
show_cache(cache, ['sys0', 'sys1', 'sys2', 'unit0', 'unit1'], 'initial cache')

## 3. The System Prompt Update Operation

The update does three things:

1. Find the old protected system span.
2. Slice off the later conversation unit cache.
3. Rebuild the new system span, then concatenate the preserved units after it.

This is why the cache can change from:

```text
[old system length 3] [units length 2]
```

to:

```text
[new system length 5] [same units length 2]
```

In [ ]:
old_system_end = 3
units_cache = slice_cache(cache, old_system_end, cache_len(cache))
show_cache(units_cache, ['unit0', 'unit1'], 'sliced units before update')

new_system_tokens = [20, 21, 22, 23, 24]
new_system_cache = make_visible_cache(new_system_tokens, start_position=0)

# This concat demonstrates the layout change. Real code also reindexes RoPE keys when positions move.
updated_cache_without_rope = concat_caches(new_system_cache, units_cache)
show_cache(
    updated_cache_without_rope,
    ['new_sys0', 'new_sys1', 'new_sys2', 'new_sys3', 'new_sys4', 'unit0', 'unit1'],
    'updated layout before RoPE reindex idea',
)

## 4. Why Position Reindexing Exists

MiniCPM-style LLMs use rotary positional embeddings, usually called RoPE. In plain terms: the key vectors contain position information. If unit tokens used to start at position `3` but now start at position `5`, their cached keys need to be rotated from old positions to new positions.

The repo does this in `StreamDecoder._reindex_rope_for_cache()`.

In [ ]:
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)

def apply_tiny_rope(x, positions, theta=10000.0):
    """Small RoPE helper for demonstration, not a drop-in replacement for every model."""
    dim = x.shape[-1]
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=x.dtype) / dim))
    freqs = torch.outer(positions.to(x.dtype), inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    cos = emb.cos().view(1, 1, len(positions), dim)
    sin = emb.sin().view(1, 1, len(positions), dim)
    return (x * cos) + (rotate_half(x) * sin)

def rope_reindex_keys(keys, old_start, new_start, length):
    old_positions = torch.arange(old_start, old_start + length)
    new_positions = torch.arange(new_start, new_start + length)

    # Approximate the idea used by the real method:
    # remove old rotation, then apply new rotation.
    unrotated = apply_tiny_rope(keys, old_positions, theta=10000.0)
    reindexed = apply_tiny_rope(unrotated, new_positions, theta=10000.0)
    return reindexed

old_unit_keys = units_cache[0][0]
moved_unit_keys = rope_reindex_keys(old_unit_keys, old_start=3, new_start=5, length=2)

print('old unit key at old position 3:', old_unit_keys[0, 0, 0].tolist())
print('same unit key after position move 3 -> 5:', moved_unit_keys[0, 0, 0].tolist())
print('\nNotice: values are unchanged by RoPE reindexing; only keys carry this positional rotation.')

## 5. Run The Repo's Actual Update Method With A Fake LLM

This cell imports `StreamDecoder` from your repo and runs `update_system_prompt()` without loading real model weights. The fake LLM is deliberately simple:

- `key = embedding + position`
- `value = embedding - position`

That makes it easy to inspect which parts were rebuilt and which parts were preserved.

In [ ]:
if not REPO_PATH.exists():
    raise RuntimeError('Set REPO_PATH to your cloned/uploaded repo before running this cell.')

from minicpmo_demo.model.runtime.stream_decoder import StreamDecoder

class FakeTokenizer:
    eos_token_id = 0
    unk_token_id = -1
    all_special_ids = []
    all_special_tokens = []

    def convert_tokens_to_ids(self, token):
        return {'<|chunk_eos|>': 1, '<|chunk_tts_eos|>': 2, '<|turn_eos|>': 3, '<|speak|>': 4}.get(token, 99)

    def encode(self, text, add_special_tokens=False):
        return [ord(c) % 97 for c in text]

    def decode(self, token_ids, skip_special_tokens=False):
        return ''.join(str(i) for i in token_ids)

class FakeEmbeddingModel:
    def __init__(self, hidden_size):
        self.embedding = torch.nn.Embedding(256, hidden_size)
        with torch.no_grad():
            weight = torch.arange(256 * hidden_size, dtype=torch.float32).reshape(256, hidden_size)
            self.embedding.weight.copy_(weight / 1000.0)

    def embed_tokens(self, token_ids):
        return self.embedding(token_ids)

class FakeLLM:
    device = torch.device('cpu')

    def __init__(self, hidden_size=4):
        self.config = SimpleNamespace(hidden_size=hidden_size, rope_theta=10000.0)
        self.model = FakeEmbeddingModel(hidden_size)

    def __call__(self, inputs_embeds, position_ids, past_key_values=None, use_cache=True, return_dict=True):
        del use_cache, return_dict
        positions = position_ids.to(inputs_embeds.dtype).unsqueeze(-1)
        keys = inputs_embeds.unsqueeze(1) + positions
        values = inputs_embeds.unsqueeze(1) - positions
        current = ((keys, values),)
        if past_key_values is None:
            cache = current
        else:
            cache = ((torch.cat([past_key_values[0][0], current[0][0]], dim=2), torch.cat([past_key_values[0][1], current[0][1]], dim=2)),)
        return SimpleNamespace(past_key_values=cache)

def make_repo_style_cache(length, hidden_size=4):
    values = torch.arange(length * hidden_size, dtype=torch.float32).reshape(1, 1, length, hidden_size)
    keys = values + 0.5
    return ((keys, values),)

decoder = StreamDecoder(FakeLLM(), FakeTokenizer())

# Initial layout: [prefix length 2] [previous length 1] [suffix length 1] [unit length 2]
decoder.cache = make_repo_style_cache(6)
decoder._preserve_prefix_length = 2
decoder._previous_token_ids = [30]
decoder._previous_content_length = 1
decoder._suffix_token_ids = [40]
decoder._system_preserve_length = 4
decoder._unit_history = [{'unit_id': 0, 'length': 2, 'type': 'audio'}]

old_unit_values = decoder.cache[0][1][:, :, 4:6, :].clone()

show_cache(decoder.cache, ['prefix0', 'prefix1', 'previous', 'suffix', 'unit0', 'unit1'], 'repo decoder before update')

In [ ]:
ref_audio_embeds = torch.ones(2, 4)

updated = decoder.update_system_prompt(
    new_prefix_token_ids=[10, 11, 12],
    new_suffix_token_ids=[50],
    new_ref_audio_embeds=ref_audio_embeds,
)

print('updated:', updated)
print('new cache length:', decoder.get_cache_length())
print('new protected prefix length:', decoder._preserve_prefix_length)
print('previous length:', decoder._previous_content_length)
print('suffix ids:', decoder._suffix_token_ids)
print('system preserve length:', decoder._system_preserve_length)

show_cache(
    decoder.cache,
    ['new_prefix0', 'new_prefix1', 'new_prefix2', 'ref0', 'ref1', 'previous', 'new_suffix', 'unit0', 'unit1'],
    'repo decoder after update',
)

print('\nPreserved unit values moved from old positions 4:6 to new positions 7:9:')
print(torch.equal(decoder.cache[0][1][:, :, 7:9, :], old_unit_values))
print('old unit values:')
print(old_unit_values[0, 0])
print('new unit values at the end:')
print(decoder.cache[0][1][0, 0, 7:9])

## 6. What This Proves And What It Does Not Prove

This proves the current implementation can surgically rebuild the protected prompt span while preserving later unit cache values. It also verifies the bookkeeping fields that tell the decoder where prefix, previous summary, suffix, and unit cache sections live.

It does not prove mathematical equivalence to replaying the entire conversation from scratch under the new system prompt. Preserved unit keys/values were originally computed while attending to the old prompt. The update keeps them for speed and continuity, then future tokens attend to the new prompt plus the preserved unit cache.

Exact recaching would require storing replayable raw inputs for every prior unit and running them again after the new prompt. That would be slower and use more memory/storage, but it is the route if exact replay semantics become necessary.

## 7. Optional: Run The Lightweight Repo Test In Colab

After the notebook cells above make sense, run the real test file. It uses the same fake-model idea, but as an automated regression check.

In [ ]:
if REPO_PATH.exists():
    %cd {REPO_PATH}
    !PYTHONPATH=src python -m pytest -q tests/test_duplex_system_prompt_update.py -s
else:
    print('Clone/upload the repo first, then run this cell.')

## 8. Optional: Real Model Smoke Shape

Only run this when your Colab runtime has the real model dependencies, model path, and enough GPU memory. This is intentionally a skeleton so you can fill in your actual model paths.

In [ ]:
# Optional real-model sketch. Leave disabled until your paths and deps are ready.
RUN_REAL_MODEL = False

if RUN_REAL_MODEL:
    from minicpmo_demo.core.processors.unified import UnifiedProcessor

    MODEL_PATH = '/content/path/to/MiniCPM-o-model'
    PT_PATH = '/content/path/to/model.pt'
    REF_AUDIO = '/content/path/to/ref.wav'

    processor = UnifiedProcessor.from_pretrained(model_path=MODEL_PATH, pt_path=PT_PATH)
    duplex = processor.set_duplex_mode(ref_audio_path=REF_AUDIO)
    duplex.prepare(system_prompt_text='You are concise.')

    before = processor.kv_cache_length
    ok = duplex.update_system_prompt(system_prompt_text='You are concise and mention tool changes when relevant.')
    after = processor.kv_cache_length

    print({'updated': ok, 'cache_before': before, 'cache_after': after})
else:
    print('Set RUN_REAL_MODEL = True only after configuring real model paths.')